# Animate a play's window

Replays the tracking window `scripts/build_windows.py` built for a single make, in the same visual style as `scripts/vendor/`'s play animation -- real team colors, jersey numbers, a home/visitor roster table, the same court image. The title is the play's own play-by-play description.

Two reference notes sit above the court and turn from gray to colored (bold) once the animation's current frame reaches them:
- **play-by-play time** (red): when `clock_s` -- the play-by-play's own recorded time -- is reached.
- **detected crossing** (green): when the make's detected rim-crossing time (`data/derived/make_times.parquet`) is reached.

Only makes with a built window (an `included=True`, or an excluded-but-still-bounded `too short`/`coverage gap` row in `data/windows/index.parquet`) can be animated -- see `scripts/animate_window.py`'s docstring. Run this with the project's `.venv` as the kernel.

In [ ]:
import sys
from pathlib import Path

import matplotlib
import pandas as pd
from IPython.display import HTML

# windows run up to windows.max_window_s (24s) at 25 Hz -- the default 20MB
# embed_limit silently truncates an animation's frames above that size.
matplotlib.rcParams["animation.embed_limit"] = 200

# scripts/ isn't an installed package -- make it (and scripts/vendor/, for
# Team.py) importable the same way the scripts themselves reach src/.
sys.path.insert(0, str(Path.cwd().parent / "scripts"))

from animate_window import build_event_animation
from nba_assists.config import get_config

cfg = get_config()
pd.set_option("display.max_colwidth", 80)

## Pick a play

A few included makes to choose from -- edit the filter (e.g. by `game_id`) to browse a specific game, or just change `GAME_ID`/`EVENTNUM` below directly.

In [ ]:
index = pd.read_parquet(cfg.get_path("windows") / "index.parquet")
makes = pd.read_parquet(cfg.get_path("derived") / "makes.parquet", columns=["game_id", "eventnum", "description"])

included = index[index["included"]].merge(makes, on=["game_id", "eventnum"])
included[["game_id", "eventnum", "description", "length_s"]].head(10)

In [ ]:
num = 6
GAME_ID = included.iloc[num]["game_id"]
EVENTNUM = int(included.iloc[num]["eventnum"])
print(GAME_ID, EVENTNUM)

## Animate it

In [ ]:
anim = build_event_animation(GAME_ID, EVENTNUM, cfg=cfg)
HTML(anim.to_jshtml())